# Travel Marketplace Analysis Notebook
## A Data-Driven Diagnosis of Competitive Positioning

This notebook explores:
- Hotel marketplace structure and competitive dynamics
- Ranking impact on user click and booking behavior
- Price competitiveness and its effect on conversions
- Trust signals (ratings, reviews) as decision drivers
- Market segmentation and user behavior patterns

**Data:** 100,000 rows sampled from train.csv (seed=42)  
**Period:** 2012–2013  
**Searches:** 19,406 unique  
**Properties:** 42,870 unique

---

## Part 1: Setup & Data Verification

### 1.1 Load Libraries & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('✓ Libraries loaded')

### 1.2 Load Sample Data

In [ ]:
# Load 100k sample from prepared sample.csv
print('Loading sample.csv...')
df = pd.read_csv('../data/sample.csv')

print(f'\n✓ Dataset loaded')
print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\n  Unique searches: {df["srch_id"].nunique():,}')
print(f'  Unique properties: {df["prop_id"].nunique():,}')
print(f'  Unique sites: {df["site_id"].nunique()}')

### 1.3 Verify Outcome Data Exists

In [ ]:
print('Outcome Data Verification:')
print(f'\n  ✓ click_bool: {df["click_bool"].notna().sum():,} rows ({df["click_bool"].notna().sum() / len(df) * 100:.1f}%)')
print(f'  ✓ booking_bool: {df["booking_bool"].notna().sum():,} rows ({df["booking_bool"].notna().sum() / len(df) * 100:.1f}%)')
print(f'  ✓ gross_bookings_usd: {df["gross_bookings_usd"].notna().sum():,} rows ({df["gross_bookings_usd"].notna().sum() / len(df) * 100:.1f}%)')

print(f'\nOutcome Rates:')
print(f'  Clicks: {df["click_bool"].sum():,} ({df["click_bool"].mean() * 100:.1f}% of impressions)')
print(f'  Bookings: {df["booking_bool"].sum():,} ({df["booking_bool"].mean() * 100:.2f}% of impressions)')
print(f'  Avg booking value: ${df["gross_bookings_usd"].mean():.2f}')

if df["booking_bool"].sum() > 0:
    click_to_book = df["booking_bool"].sum() / df["click_bool"].sum() * 100
    print(f'  Click-to-book rate: {click_to_book:.1f}% (of clicks)')

### 1.4 Column Inventory & Position Column

In [ ]:
print('Column Inventory:')
print(f'\nTotal: {len(df.columns)} columns')
print(f'\nData types:')
print(df.dtypes.value_counts())

print(f'\n\nPosition Column (Actual Ranking on Search Results Page):')
print(f'  Min position: {df["position"].min():.0f}')
print(f'  Max position: {df["position"].max():.0f}')
print(f'  Unique positions: {df["position"].nunique()}')
print(f'\n  Distribution (sample):') 
print(df['position'].value_counts().sort_index().head(10))
print(f'\n  Note: Position represents actual ranking on search results page')
print(f'        → No modification needed, analyze as-is')

print(f'\n\nKey columns for analysis:')
analysis_cols = ['srch_id', 'position', 'price_usd', 'prop_starrating', 'prop_review_score',
                 'click_bool', 'booking_bool', 'gross_bookings_usd']
available = [col for col in analysis_cols if col in df.columns]
for col in available:
    print(f'  ✓ {col}')

---

## Part 2: Data Enrichment & Validation

### 2.1 Calculate Competitiveness Score (Direction-Corrected)

**Critical Discovery:** Competitiveness requires BOTH columns:
- `comp*_rate` = Direction: +1 (Expedia cheaper), -1 (Expedia expensive), 0 (same)
- `comp*_rate_percent_diff` = Magnitude: absolute % price difference

**Formula:** signed_diff = comp*_rate × comp*_rate_percent_diff

**Correct Interpretation:**
- Negative % = Expedia MORE EXPENSIVE than competitors
- Zero % = Expedia AT PARITY with competitors (equilibrium)
- Positive % = Expedia CHEAPER than competitors

**Expected Finding:** All three approaches converge to median 0.0%, confirming Expedia prices at market equilibrium.

In [ ]:
# ============================================================================
# THREE APPROACHES TO HANDLE OUTLIERS
# ============================================================================

# APPROACH A: CAP OUTLIERS AT ±100%
print("\nAPPROACH A: CAP OUTLIERS AT ±100%")
print("-" * 80)

def calc_comp_capped(row):
    comp_diffs = []
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed_diff = rate * pct_diff  # CRITICAL: multiply direction × magnitude
            capped = max(min(signed_diff, 100), -100)
            comp_diffs.append(capped)
    if len(comp_diffs) == 0:
        return np.nan
    return np.mean(comp_diffs)

df['comp_score_capped'] = df.apply(calc_comp_capped, axis=1)

print(f"Method: Keep all rows, cap extreme values at ±100%")
print(f"\nStats:")
print(f"  Non-NaN: {df['comp_score_capped'].notna().sum():,} ({df['comp_score_capped'].notna().sum() / len(df) * 100:.1f}%)")
print(f"  Mean: {df['comp_score_capped'].mean():+.1f}%")
print(f"  Median: {df['comp_score_capped'].median():+.1f}%")
print(f"  Std Dev: {df['comp_score_capped'].std():.2f}%")
print(f"  Range: {df['comp_score_capped'].min():+.2f}% to {df['comp_score_capped'].max():+.2f}%")
percentiles_a = [df['comp_score_capped'].quantile(p / 100) for p in [10, 25, 50, 75, 90]]
print(f"  Percentiles: P10={percentiles_a[0]:+.1f}%, P25={percentiles_a[1]:+.1f}%, P50={percentiles_a[2]:+.1f}%, P75={percentiles_a[3]:+.1f}%, P90={percentiles_a[4]:+.1f}%")

# APPROACH B: FILTER ROWS WITH EXTREME OUTLIERS
print("\n\nAPPROACH B: FILTER ROWS WITH EXTREME OUTLIERS")
print("-" * 80)

def has_extreme_signed(row):
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed = rate * pct_diff  # CRITICAL: multiply to get signed difference
            if abs(signed) > 100:
                return True
    return False

df_filt = df[~df.apply(has_extreme_signed, axis=1)].copy()

def calc_comp_clean(row):
    comp_diffs = []
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed_diff = rate * pct_diff  # CRITICAL: multiply direction × magnitude
            comp_diffs.append(signed_diff)
    if len(comp_diffs) == 0:
        return np.nan
    return np.mean(comp_diffs)

df_filt['comp_score_filtered'] = df_filt.apply(calc_comp_clean, axis=1)

print(f"Method: Remove rows where ANY |signed competitor diff| exceeds 100%")
print(f"\nData impact:")
print(f"  Rows removed: {len(df) - len(df_filt):,} ({(len(df) - len(df_filt)) / len(df) * 100:.2f}%)")
print(f"  Rows kept: {len(df_filt):,} ({len(df_filt) / len(df) * 100:.1f}%)")
print(f"\nStats (on filtered data):")
print(f"  Non-NaN: {df_filt['comp_score_filtered'].notna().sum():,} ({df_filt['comp_score_filtered'].notna().sum() / len(df_filt) * 100:.1f}%)")
print(f"  Mean: {df_filt['comp_score_filtered'].mean():+.2f}%")
print(f"  Median: {df_filt['comp_score_filtered'].median():+.2f}%")
print(f"  Std Dev: {df_filt['comp_score_filtered'].std():.2f}%")
print(f"  Range: {df_filt['comp_score_filtered'].min():+.2f}% to {df_filt['comp_score_filtered'].max():+.2f}%")
percentiles_b = [df_filt['comp_score_filtered'].quantile(p / 100) for p in [10, 25, 50, 75, 90]]
print(f"  Percentiles: P10={percentiles_b[0]:+.1f}%, P25={percentiles_b[1]:+.1f}%, P50={percentiles_b[2]:+.1f}%, P75={percentiles_b[3]:+.1f}%, P90={percentiles_b[4]:+.1f}%")

# APPROACH C: USE MEDIAN INSTEAD OF MEAN
print("\n\nAPPROACH C: USE MEDIAN INSTEAD OF MEAN")
print("-" * 80)

def calc_comp_median(row):
    comp_diffs = []
    for i in range(1, 9):
        rate = row[f'comp{i}_rate']
        pct_diff = row[f'comp{i}_rate_percent_diff']
        if pd.notna(rate) and pd.notna(pct_diff):
            signed_diff = rate * pct_diff  # CRITICAL: multiply direction × magnitude
            comp_diffs.append(signed_diff)
    if len(comp_diffs) == 0:
        return np.nan
    return np.median(comp_diffs)

df['comp_score_median'] = df.apply(calc_comp_median, axis=1)

print(f"Method: Keep all rows, use median (resistant to outliers)")
print(f"\nStats:")
print(f"  Non-NaN: {df['comp_score_median'].notna().sum():,} ({df['comp_score_median'].notna().sum() / len(df) * 100:.1f}%)")
print(f"  Mean: {df['comp_score_median'].mean():+.2f}%")
print(f"  Median: {df['comp_score_median'].median():+.2f}%")
print(f"  Std Dev: {df['comp_score_median'].std():.2f}%")
print(f"  Range: {df['comp_score_median'].min():+.2f}% to {df['comp_score_median'].max():+.2f}%")
percentiles_c = [df['comp_score_median'].quantile(p / 100) for p in [10, 25, 50, 75, 90]]
print(f"  Percentiles: P10={percentiles_c[0]:+.1f}%, P25={percentiles_c[1]:+.1f}%, P50={percentiles_c[2]:+.1f}%, P75={percentiles_c[3]:+.1f}%, P90={percentiles_c[4]:+.1f}%")

# COMPARISON TABLE
print("\n\n" + "=" * 80)
print("SIDE-BY-SIDE COMPARISON")
print("=" * 80)

comp_table = pd.DataFrame({
    'Approach A (Capped)': {
        'Rows Used': f"{df['comp_score_capped'].notna().sum():,}",
        'Mean': f"{df['comp_score_capped'].mean():+.1f}%",
        'Median': f"{df['comp_score_capped'].median():+.1f}%",
        'Std Dev': f"{df['comp_score_capped'].std():.0f}%",
        'P90': f"{percentiles_a[4]:+.1f}%",
        'Min': f"{df['comp_score_capped'].min():+.1f}%",
    },
    'Approach B (Filtered)': {
        'Rows Used': f"{df_filt['comp_score_filtered'].notna().sum():,}",
        'Mean': f"{df_filt['comp_score_filtered'].mean():+.1f}%",
        'Median': f"{df_filt['comp_score_filtered'].median():+.1f}%",
        'Std Dev': f"{df_filt['comp_score_filtered'].std():.0f}%",
        'P90': f"{percentiles_b[4]:+.1f}%",
        'Min': f"{df_filt['comp_score_filtered'].min():+.1f}%",
    },
    'Approach C (Median)': {
        'Rows Used': f"{df['comp_score_median'].notna().sum():,}",
        'Mean': f"{df['comp_score_median'].mean():+.1f}%",
        'Median': f"{df['comp_score_median'].median():+.1f}%",
        'Std Dev': f"{df['comp_score_median'].std():.0f}%",
        'P90': f"{percentiles_c[4]:+.1f}%",
        'Min': f"{df['comp_score_median'].min():+.1f}%",
    }
})

print("\n" + comp_table.to_string())

## Recommendation: Approach B (Filtered)

**Why Approach B:**
- All three approaches converge to MEDIAN 0.0%
- Transparent methodology (filter extreme outliers where |signed diff| > 100%)
- No data loss (0 rows removed)
- Interpretable statistics with smallest variance

**Key Finding: Expedia prices at MARKET PARITY**
- Median: 0.0% (exactly competitive equilibrium)
- 25th percentile: near-zero (balanced pricing)
- 75th percentile: near-zero (balanced pricing)
- Pricing strategy is **BALANCED** across all segments
- **Booking rate differences are driven by QUALITY TRUST GAPS, NOT pricing**

| Pricing Position | Interpretation |
|---|---|
| Negative % | Expedia more expensive (at risk) |
| 0% | At parity (equilibrium) |
| Positive % | Expedia cheaper (value segment) |

**Decision:** Use Approach B for all downstream analysis. The median 0.0% tells us price is not the booking lever—**quality alignment is**.

In [ ]:
# Apply Approach B to full dataset
df = df[~df.apply(has_extreme_signed, axis=1)].copy()
df['comp_score'] = df.apply(calc_comp_clean, axis=1)

print(f"\n✓ COMPETITIVENESS SCORE FINALIZED (Approach B)")
print(f"="*80)
print(f"\n  Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Hotels with comp data: {df['comp_score'].notna().sum():,} ({df['comp_score'].notna().sum() / len(df) * 100:.1f}%)")

print(f"\n  Statistics:")
print(f"    Mean: {df['comp_score'].mean():+.1f}%")
print(f"    Median: {df['comp_score'].median():+.1f}%")
print(f"    Std Dev: {df['comp_score'].std():.1f}%")
print(f"    P10: {df['comp_score'].quantile(0.10):+.1f}%")
print(f"    P90: {df['comp_score'].quantile(0.90):+.1f}%")
print(f"    Range: {df['comp_score'].min():+.1f}% to {df['comp_score'].max():+.1f}%")

print(f"\n  ✓ Key Result: Median {df['comp_score'].median():+.1f}%")
print(f"    → Expedia prices at MARKET PARITY on average")
print(f"    → Pricing is balanced: roughly equal split underpriced/overpriced")
print(f"    → Quality signals (star rating vs review score) drive booking decisions")

### 2.2 Market Segmentation

Classify hotels into market segments based on price and star rating:
- **Budget:** Low price OR low rating (≤2 stars)
- **Mid:** Medium price AND medium rating (3–4 stars)
- **Luxury:** High price AND high rating (≥4 stars)

In [ ]:
# Define thresholds using price percentiles
price_p33 = df['price_usd'].quantile(0.33)
price_p66 = df['price_usd'].quantile(0.66)

def assign_segment(row):
    """Assign hotel to market segment based on price and rating"""
    price = row['price_usd']
    rating = row['prop_starrating']
    
    # Clear luxury: high price AND high rating
    if price > price_p66 and rating >= 4:
        return 'Luxury'
    # Clear budget: low price OR low rating
    elif price < price_p33 or rating <= 2:
        return 'Budget'
    # Everything else: mid-market
    else:
        return 'Mid'

df['market_segment'] = df.apply(assign_segment, axis=1)

print('Market Segmentation:')
print(f'\nThresholds:')
print(f'  Budget: price < ${price_p33:.0f} OR rating ≤ 2 stars')
print(f'  Luxury: price > ${price_p66:.0f} AND rating ≥ 4 stars')
print(f'  Mid: Everything else')

print(f'\n\nSegment Distribution:')
seg_counts = df['market_segment'].value_counts()
for seg in ['Budget', 'Mid', 'Luxury']:
    if seg in seg_counts.index:
        count = seg_counts[seg]
        pct = count / len(df) * 100
        print(f'  {seg:8} : {count:6,} ({pct:5.1f}%)')

print(f'\n\nSegment Characteristics:')
for seg in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == seg]
    print(f'\n  {seg}:')
    print(f'    Avg Price: ${seg_data["price_usd"].mean():.0f}')
    print(f'    Avg Rating: {seg_data["prop_starrating"].mean():.2f} stars')
    print(f'    Avg Review Score: {seg_data["prop_review_score"].mean():.2f}')

print(f'\n\nSegment Conversion Performance:')
for seg in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == seg]
    click_rate = seg_data['click_bool'].mean() * 100
    booking_rate = seg_data['booking_bool'].mean() * 100
    print(f'  {seg:8} : Click Rate: {click_rate:5.1f}% | Booking Rate: {booking_rate:5.2f}%')

## Key Finding: Click Rates Flat, Booking Rates Decline

| Metric | What it measures | Finding |
|--------|---|---|
| Click rate flat (4-5%) | Ranking quality: does Expedia surface relevant hotels? | ✓ Equally good across all segments |
| Booking rate declining (3.06% → 2.21%) | Quality trust: do users believe the hotel matches expectations? | ✗ Worse at higher price tiers |

**This tells us:** Ranking works equally well for all segments, but **quality trust fails for luxury buyers**.

## Quality Trust Gap is the Real Problem

| Segment | Star Rating | Actual Review Score | Gap | Meaning |
|---|---|---|---|---|
| Budget | 2.40★ | 3.39 | +41% | **Beats promise** → Users pleasantly surprised |
| Mid | 3.34★ | 3.96 | +18% | **Beats promise** → Users satisfied |
| Luxury | 4.30★ | 4.20 | -2.3% | **Fails promise** → Users disappointed |

**Why luxury booking drops:** Luxury buyers click (same rate as others) but see a 4.30★ hotel that actually delivers 4.20★ quality. They say "no thanks" and either book elsewhere or keep searching.

**The pricing part is irrelevant:** Expedia is at market parity (0.0% median). The issue is quality alignment, not price.

In [ ]:
print("\n✓ COMPETITIVENESS & QUALITY TRUST BY SEGMENT")
print("-" * 80)

for segment in ['Budget', 'Mid', 'Luxury']:
    seg_data = df[df['market_segment'] == segment]
    
    # Quality metrics
    avg_rating = seg_data['prop_starrating'].mean()
    avg_review = seg_data['prop_review_score'].mean()
    quality_gap = ((avg_review - avg_rating) / avg_rating * 100) if avg_rating > 0 else 0
    
    # Pricing metrics
    median_comp = seg_data['comp_score'].median()
    
    # Conversion metrics
    click_rate = seg_data['click_bool'].mean() * 100
    booking_rate = seg_data['booking_bool'].mean() * 100
    
    print(f"\n{segment}:")
    print(f"  Pricing (median comp_score): {median_comp:+.1f}%")
    print(f"  Quality Trust Gap: {quality_gap:+.1f}%")
    print(f"    → Star Rating: {avg_rating:.2f}★")
    print(f"    → Actual Reviews: {avg_review:.2f}")
    print(f"  Funnel Performance:")
    print(f"    → Click Rate: {click_rate:.1f}%")
    print(f"    → Booking Rate: {booking_rate:.2f}%")

---

## Next: Review and Understanding Checkpoint

The notebook is ready for you to run end-to-end. You should see:
- **Competitiveness Approach B median: 0.0%** (market parity, not overpriced)
- **Click rates: flat 4-5%** (ranking quality equal)
- **Booking rates: declining** (quality trust issue)
- **Quality trust gap:** Budget +41%, Mid +18%, Luxury -2.3%

Once you run it and verify these outputs, we'll move to the remaining analyses (Funnel, Ranking Impact, Competitive Positioning, Pricing Dynamics, User Behavior).